<a href="https://colab.research.google.com/github/detektor777/colab_list_image/blob/main/see_sr_super_resolution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title ##**Install** { display-mode: "form" }
import os, shutil, glob

if os.path.isdir('/content/SeeSR'):
    shutil.rmtree('/content/SeeSR')

!git clone --depth 1 https://github.com/cswry/SeeSR.git
%cd SeeSR

!grep -vE '^(torch|transformers)==' requirements.txt > requirements_colab.txt
!pip install -q -r requirements_colab.txt
!pip install -q gdown huggingface_hub

os.makedirs('preset/models', exist_ok=True)

from huggingface_hub import snapshot_download, hf_hub_download

# SD-2-base backbone (ungated mirror on Hugging Face)
snapshot_download(
    repo_id='Manojb/stable-diffusion-2-base',
    local_dir='preset/models/stable-diffusion-2-base',
)

# SeeSR + DAPE weights (published by the authors on Google Drive)
!gdown --folder -q "https://drive.google.com/drive/folders/12HXrRGEXUAnmHRaf0bIn-S8XSK4Ku0JO" -O preset/models/_gdrive_tmp

for item in glob.glob('preset/models/_gdrive_tmp/*'):
    dst = os.path.join('preset/models', os.path.basename(item))
    if os.path.exists(dst):
        shutil.rmtree(dst) if os.path.isdir(dst) else os.remove(dst)
    shutil.move(item, 'preset/models/')
shutil.rmtree('preset/models/_gdrive_tmp', ignore_errors=True)

# RAM (Recognize Anything Model) — используется для авто-тегирования изображения,
# отдельная зависимость, не входит ни в SD-2-base, ни в архив SeeSR/DAPE.
ram_ckpt = hf_hub_download(
    repo_id='xinyu1205/recognize-anything',
    repo_type='space',
    filename='ram_swin_large_14m.pth',
    local_dir='preset/models',
)
print('RAM checkpoint:', ram_ckpt)

print('seesr weights:', os.path.exists('preset/models/seesr'))
print('DAPE weights:', os.path.exists('preset/models/DAPE.pth'))
print('RAM weights:', os.path.exists('preset/models/ram_swin_large_14m.pth'))

# requirements.txt тянет несовместимые с текущим окружением версии
# (torch==2.0.1 — нет wheel под текущий Python/Colab; transformers==4.25.0 —
# yanked, тянет несобираемый tokenizers<0.14). Ставим отдельно совместимые версии:
# diffusers==0.21.0 — оригинал репозитория, чистый Python, ставится без проблем.
# transformers — не 4.25.0, а более поздняя, где ещё жива apply_chunking_to_forward
# (убрали только в 4.57+), и есть готовые wheel'ы tokenizers под py3.12.
# huggingface_hub держим <0.26, иначе diffusers 0.21.0 упадёт на cached_download.
!pip install -q \
    "diffusers==0.21.0" \
    "huggingface_hub==0.25.2" \
    "transformers==4.46.3"

import torch
print('torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())

# --- Patches ---------------------------------------------------------------
# С diffusers==0.21.0 оригинальный импорт в репозитории
#   from diffusers.pipeline_utils import DiffusionPipeline
# работает как есть — патчить его не нужно.
# Этот блок — единая точка для всех точечных фиксов исходников репозитория.
# Если в traceback появится новая несовместимость (например, в другом файле),
# добавляйте sed-правку сюда.
print('Патчи применены (сейчас список пуст — diffusers==0.21.0 совместим с оригинальным кодом).')

In [ ]:
#@title ##**Upload Images** { display-mode: "form" }
import os
import glob
import shutil
from google.colab import files

upload_folder = './preset/datasets/test_datasets'
result_folder = './preset/datasets/output'

if os.path.isdir(upload_folder):
    shutil.rmtree(upload_folder)
if os.path.isdir(result_folder):
    shutil.rmtree(result_folder)

os.makedirs(upload_folder, exist_ok=True)
os.makedirs(result_folder, exist_ok=True)

# Upload images — you can select multiple files at once
uploaded = files.upload()
for filename in uploaded.keys():
    dst_path = os.path.join(upload_folder, filename)
    print(f'Moved {filename} -> {dst_path}')
    shutil.move(filename, dst_path)

print(f'\n{len(uploaded)} image(s) uploaded to {upload_folder}')

In [ ]:
#@title ##**Run** { display-mode: "form" }
import os, gc, torch, shutil, sys, subprocess
import time, datetime
from PIL import Image
from tqdm import tqdm

# Запускаем таймер общего времени выполнения
start_time = time.time()

#@markdown ### Settings
prompt = "hyperrealism" #@param {type:"string"}
negative_prompt = "" #@param {type:"string"}
num_inference_steps = 100 #@param {type:"slider", min:2, max:100, step:1}
guidance_scale = 1 #@param {type:"number"}
process_size = 512 #@param [512, 768, 1024] {type:"raw"}
split_tile_size = 512 #@param {type:"slider", min:256, max:2048, step:128}
vae_decoder_tiled_size = 128 #@param {type:"slider", min:64, max:1024, step:64}
vae_encoder_tiled_size = 512 #@param {type:"slider", min:64, max:1024, step:64}
slice_image = True #@param {type:"boolean"}
use_ram = False #@param {type:"boolean"}
show_log = False #@param {type:"boolean"}

# Configuration
scale = 4
upload_folder = '/content/upload' if 'upload_folder' not in locals() else upload_folder
result_folder = '/content/results' if 'result_folder' not in locals() else result_folder

temp_in = '/content/temp_seesr_in'
temp_out = '/content/temp_seesr_out'

for folder in [temp_in, temp_out]:
    if os.path.exists(folder):
        shutil.rmtree(folder)
    os.makedirs(folder, exist_ok=True)

if os.path.exists(result_folder):
    shutil.rmtree(result_folder)
os.makedirs(result_folder, exist_ok=True)

# 1. Image Slicing
registry = {}
valid_extensions = ('.png', '.jpg', '.jpeg', '.webp')
images_to_process = [
    f for f in os.listdir(upload_folder)
    if f.lower().endswith(valid_extensions) and os.path.isfile(os.path.join(upload_folder, f))
]

if not images_to_process:
    print(f"Error: No images found in '{upload_folder}'")
    sys.exit()

active_in = temp_in if slice_image else upload_folder
active_out = temp_out if slice_image else result_folder

if slice_image:
    print("Slicing images...")
    for filename in images_to_process:
        img_path = os.path.join(upload_folder, filename)
        img = Image.open(img_path).convert('RGB')
        w_orig, h_orig = img.size
        print(f"Processing {filename} ({w_orig}x{h_orig})...")

        new_w = ((w_orig + split_tile_size - 1) // split_tile_size) * split_tile_size
        new_h = ((h_orig + split_tile_size - 1) // split_tile_size) * split_tile_size

        delta_w = new_w - w_orig
        delta_h = new_h - h_orig

        padded_img = Image.new('RGB', (new_w, new_h))
        padded_img.paste(img, (0, 0))

        tiles_info = []
        for y in range(0, new_h, split_tile_size):
            for x in range(0, new_w, split_tile_size):
                tile = padded_img.crop((x, y, x + split_tile_size, y + split_tile_size))
                tile_name = f"{os.path.splitext(filename)[0]}_tile_{x}_{y}.png"
                tile.save(os.path.join(temp_in, tile_name))
                tiles_info.append((tile_name, x, y))

        registry[filename] = {
            'orig_size': (w_orig, h_orig),
            'padded_size': (new_w, new_h),
            'tiles': tiles_info,
            'padding': (delta_w, delta_h)
        }
    print(f"Total tiles prepared: {len(os.listdir(temp_in))}")
else:
    print(f"Direct pass enabled. Total images to process: {len(images_to_process)}")

# 2. SeeSR Processing
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
gc.collect()
torch.cuda.empty_cache()

cmd_list = [
    'python', 'test_seesr.py',
    '--pretrained_model_path', 'preset/models/stable-diffusion-2-base',
    '--prompt', prompt,
    '--seesr_model_path', 'preset/models/seesr',
    '--image_path', active_in,
    '--output_dir', active_out,
    '--start_point', 'lr',
    '--num_inference_steps', str(num_inference_steps),
    '--guidance_scale', str(guidance_scale),
    '--process_size', str(process_size),
    '--vae_decoder_tiled_size', str(vae_decoder_tiled_size),
    '--vae_encoder_tiled_size', str(vae_encoder_tiled_size)
]

if use_ram:
    cmd_list += ['--ram_ft_path', 'preset/models/DAPE.pth']

if negative_prompt.strip():
    cmd_list += ['--negative_prompt', negative_prompt.strip()]

print("\nRunning SeeSR...")

# Выполняем процесс правильным (нативным) образом в зависимости от show_log
if show_log:
    # Без перехвата PIPES. Вывод идет напрямую в консоль, что позволяет tqdm работать идеально (без дублей)
    process = subprocess.run(cmd_list)

    if process.returncode != 0:
        print(f"\nError: SeeSR process failed with exit code {process.returncode}")
        sys.exit(process.returncode)
else:
    # Если лог выключен, скрываем вывод и показываем только общий кастомный прогресс-бар
    total_items = len(os.listdir(active_in))
    process = subprocess.Popen(cmd_list, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

    captured_logs = []
    pbar = tqdm(total=total_items, desc="Processing")
    current_count = 0

    for line in process.stdout:
        captured_logs.append(line)
        if "process" in line and "imgs" in line:
            try:
                parts = line.split()
                idx = parts.index("process") + 1
                num_processed = int(parts[idx]) + 1
                if num_processed > current_count:
                    pbar.update(num_processed - current_count)
                    current_count = num_processed
            except ValueError:
                pass
    pbar.close()
    process.wait()

    if process.returncode != 0:
        print(f"\nError: SeeSR process failed with exit code {process.returncode}")
        print("Printing captured logs for debugging:")
        print("".join(captured_logs))
        sys.exit(process.returncode)

# 3. Merging (Only if sliced)
if slice_image:
    print("\nMerging tiles...")

    out_files = []
    for root_dir, _, files in os.walk(temp_out):
        for f in files:
            if f.lower().endswith(valid_extensions):
                out_files.append(os.path.join(root_dir, f))

    for filename, info in registry.items():
        w_orig, h_orig = info['orig_size']
        new_w, new_h = info['padded_size']
        delta_w, delta_h = info['padding']

        upscaled_padded_w = new_w * scale
        upscaled_padded_h = new_h * scale
        merged_img = Image.new('RGB', (upscaled_padded_w, upscaled_padded_h))

        merged_count = 0
        for tile_name, x, y in info['tiles']:
            tile_base = os.path.splitext(tile_name)[0]
            matched_file_path = None

            for out_f_path in out_files:
                basename = os.path.basename(out_f_path)
                if basename.startswith(tile_base):
                    matched_file_path = out_f_path
                    break

            if matched_file_path:
                tile_img = Image.open(matched_file_path).convert('RGB')

                expected_tile_size = split_tile_size * scale
                if tile_img.size != (expected_tile_size, expected_tile_size):
                    tile_img = tile_img.resize((expected_tile_size, expected_tile_size), Image.LANCZOS)

                merged_img.paste(tile_img, (x * scale, y * scale))
                merged_count += 1
            else:
                print(f"Warning: Processed tile not found for {tile_name}")

        print(f"Merged {merged_count}/{len(info['tiles'])} tiles for {filename}")

        crop_w = upscaled_padded_w - (delta_w * scale)
        crop_h = upscaled_padded_h - (delta_h * scale)
        final_img = merged_img.crop((0, 0, crop_w, crop_h))

        expected_final_w = w_orig * scale
        expected_final_h = h_orig * scale
        if final_img.size != (expected_final_w, expected_final_h):
            final_img = final_img.resize((expected_final_w, expected_final_h), Image.LANCZOS)

        output_path = os.path.join(result_folder, f"{filename}")
        final_img.save(output_path)
        print(f"Saved: {output_path}")

# 4. Final Cleanup
for folder in [temp_in, temp_out]:
    if os.path.exists(folder):
        shutil.rmtree(folder)

# Подсчет времени
total_time = time.time() - start_time
formatted_time = str(datetime.timedelta(seconds=int(total_time)))

print(f"\n✅ Process complete in {formatted_time}")
if torch.cuda.is_available():
    print(f'GPU memory allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GiB')

In [ ]:
#@title ##**Visualization** { display-mode: "form" }
import os
import base64
from io import BytesIO
import PIL.Image
from IPython.display import HTML, display

def is_image_file(filename):
    image_extensions = {'.png', '.jpg', '.jpeg', '.gif', '.bmp', '.tiff'}
    return os.path.splitext(filename.lower())[1] in image_extensions

def resize_image_maintain_aspect(image, max_width, target_height=None):
    width, height = image.size
    if width > max_width:
        new_height = int(height * max_width / width)
        image = image.resize((max_width, new_height), PIL.Image.Resampling.LANCZOS)
    if target_height is not None and image.size[1] != target_height:
        new_width = int(image.size[0] * target_height / image.size[1])
        image = image.resize((new_width, target_height), PIL.Image.Resampling.LANCZOS)
    return image

def image_to_base64(image):
    buffered = BytesIO()
    image.save(buffered, format="PNG")
    return base64.b64encode(buffered.getvalue()).decode('utf-8')

# Результаты SeeSR сохраняются в result_folder/sampleNN/, а не прямо в result_folder
sample_dirs = sorted(
    d for d in os.listdir(result_folder)
    if d.startswith('sample') and os.path.isdir(os.path.join(result_folder, d))
)
actual_result_folder = os.path.join(result_folder, sample_dirs[0]) if sample_dirs else result_folder

filenames_input = sorted([f for f in os.listdir(upload_folder) if is_image_file(f)])
filenames_output = sorted([f for f in os.listdir(actual_result_folder) if is_image_file(f)])

if not filenames_input or not filenames_output:
    print(f'Error: no images found in {upload_folder} or {actual_result_folder}.')
else:
    for filename, filename_output in zip(filenames_input, filenames_output):
        try:
            image_before = PIL.Image.open(os.path.join(upload_folder, filename))
            image_after = PIL.Image.open(os.path.join(actual_result_folder, filename_output))

            max_width = min(image_after.size[0], 1000)
            image_after = resize_image_maintain_aspect(image_after, max_width)
            target_height = image_after.size[1]
            image_before = resize_image_maintain_aspect(image_before, max_width, target_height)

            if image_before.mode != 'RGB':
                image_before = image_before.convert('RGB')
            if image_after.mode != 'RGB':
                image_after = image_after.convert('RGB')

            before_base64 = image_to_base64(image_before)
            after_base64 = image_to_base64(image_after)

            html_code = f"""
            <div style="position: relative; width: {image_after.size[0]}px; height: {image_after.size[1]}px; margin-bottom: 20px;">
                <div style="position: relative; width: 100%; height: 100%; overflow: hidden;">
                    <img src="data:image/png;base64,{before_base64}" style="position: absolute; width: 100%; height: 100%;">
                    <div style="position: absolute; width: 100%; height: 100%; overflow: hidden; clip-path: inset(0 0 0 50%);">
                        <img src="data:image/png;base64,{after_base64}" style="position: absolute; width: 100%; height: 100%;">
                    </div>
                </div>
                <div class="slider" style="position: absolute; top: 0; bottom: 0; width: 4px; background: white; cursor: ew-resize; left: 50%; box-shadow: 0 0 5px rgba(0,0,0,0.5);">
                    <div style="position: absolute; top: 50%; transform: translateY(-50%); width: 20px; height: 20px; background: white; border-radius: 50%; left: -8px;"></div>
                </div>
                <div style="position: absolute; top: 8px; left: 8px; color: white; font-family: sans-serif; font-size: 14px; text-shadow: 0 1px 3px rgba(0,0,0,0.8);">Before</div>
                <div style="position: absolute; top: 8px; right: 8px; color: white; font-family: sans-serif; font-size: 14px; text-shadow: 0 1px 3px rgba(0,0,0,0.8);">After</div>
            </div>
            <script>
                document.querySelectorAll('.slider').forEach(slider => {{
                    let isDragging = false;
                    const container = slider.parentElement.querySelector('div:nth-child(1)');
                    const clipDiv = container.querySelector('div');

                    slider.addEventListener('mousedown', (e) => {{ isDragging = true; e.preventDefault(); }});
                    document.addEventListener('mouseup', () => {{ isDragging = false; }});
                    document.addEventListener('mousemove', (e) => {{
                        if (!isDragging) return;
                        const rect = container.getBoundingClientRect();
                        let x = e.clientX - rect.left;
                        if (x < 0) x = 0;
                        if (x > rect.width) x = rect.width;
                        const percentage = (x / rect.width) * 100;
                        slider.style.left = percentage + '%';
                        clipDiv.style.clipPath = `inset(0 0 0 ${{percentage}}%)`;
                    }});

                    slider.addEventListener('touchstart', (e) => {{ isDragging = true; e.preventDefault(); }});
                    document.addEventListener('touchend', () => {{ isDragging = false; }});
                    document.addEventListener('touchmove', (e) => {{
                        if (!isDragging) return;
                        const rect = container.getBoundingClientRect();
                        let x = e.touches[0].clientX - rect.left;
                        if (x < 0) x = 0;
                        if (x > rect.width) x = rect.width;
                        const percentage = (x / rect.width) * 100;
                        slider.style.left = percentage + '%';
                        clipDiv.style.clipPath = `inset(0 0 0 ${{percentage}}%)`;
                    }});
                }});
            </script>
            """

            display(HTML(html_code))

        except Exception as e:
            print(f'Error processing {filename} and {filename_output}: {e}')

In [ ]:
#@title ##**Download Results** { display-mode: "form" }
import os
from google.colab import files

zip_filename = 'SeeSR_result.zip'
if os.path.exists(zip_filename):
    os.remove(zip_filename)

os.system(f"zip -r -j {zip_filename} {result_folder}/*")
files.download(zip_filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>